# T20 World Cup Batting Analysis

## Comprehensive analysis of batting performances across T20 World Cups (2014-2024)

This notebook explores:
- Top run scorers and their strike rates
- Batting consistency and averages
- Boundary hitting patterns
- Tournament evolution over time
- Player comparisons and rankings

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded")

In [ ]:
# Load processed data
batting_stats = pd.read_csv('../data/processed/player_batting_stats.csv')
all_deliveries = pd.read_csv('../data/processed/all_deliveries.csv')
match_summaries = pd.read_csv('../data/processed/match_summaries.csv')

print(f"📊 Loaded data:")
print(f"  - Batting stats: {len(batting_stats)} players")
print(f"  - Deliveries: {len(all_deliveries):,} balls")
print(f"  - Matches: {len(match_summaries)} matches")

## 1. Top Run Scorers Analysis

In [ ]:
# Filter players with minimum 200 runs
top_batsmen = batting_stats[batting_stats['runs'] >= 200].sort_values('runs', ascending=False).head(20)

print("🏆 Top 20 Run Scorers (min 200 runs)")
print("=" * 80)
print(top_batsmen[['player', 'runs', 'balls_faced', 'average', 'strike_rate', 'matches', 'fours', 'sixes']].to_string(index=False))

In [ ]:
# Visualization: Top 15 Run Scorers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Total runs
top_15 = batting_stats.nlargest(15, 'runs')
ax1.barh(range(len(top_15)), top_15['runs'], color='steelblue')
ax1.set_yticks(range(len(top_15)))
ax1.set_yticklabels(top_15['player'])
ax1.set_xlabel('Total Runs', fontsize=12)
ax1.set_title('Top 15 Run Scorers - T20 World Cups (2014-2024)', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# Add value labels
for i, (runs, sr) in enumerate(zip(top_15['runs'], top_15['strike_rate'])):
    ax1.text(runs + 20, i, f"{runs} ({sr:.1f})", va='center', fontsize=9)

# Strike Rate vs Average scatter
qualified = batting_stats[batting_stats['runs'] >= 200]
scatter = ax2.scatter(qualified['average'], qualified['strike_rate'], 
                     s=qualified['runs']/3, alpha=0.6, c=qualified['runs'], 
                     cmap='viridis')
ax2.set_xlabel('Batting Average', fontsize=12)
ax2.set_ylabel('Strike Rate', fontsize=12)
ax2.set_title('Strike Rate vs Average (min 200 runs)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Annotate top performers
for _, row in qualified.nlargest(5, 'runs').iterrows():
    ax2.annotate(row['player'], (row['average'], row['strike_rate']), 
                fontsize=8, alpha=0.7)

plt.colorbar(scatter, ax=ax2, label='Total Runs')
plt.tight_layout()
plt.show()

## 2. Boundary Hitting Analysis

In [ ]:
# Calculate boundary percentage
qualified = batting_stats[batting_stats['runs'] >= 200].copy()
qualified['boundary_runs'] = (qualified['fours'] * 4) + (qualified['sixes'] * 6)
qualified['boundary_pct'] = (qualified['boundary_runs'] / qualified['runs'] * 100).round(1)
qualified['boundaries_per_match'] = ((qualified['fours'] + qualified['sixes']) / qualified['matches']).round(1)

print("💥 Biggest Hitters (min 200 runs, sorted by sixes)")
print("=" * 80)
biggest_hitters = qualified.nlargest(15, 'sixes')[['player', 'runs', 'fours', 'sixes', 'boundary_pct', 'boundaries_per_match']]
print(biggest_hitters.to_string(index=False))

In [ ]:
# Visualization: Fours vs Sixes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Stacked bar: Fours and Sixes
top_boundary = qualified.nlargest(15, 'sixes')
x = range(len(top_boundary))
ax1.barh(x, top_boundary['fours'], label='Fours', color='lightcoral')
ax1.barh(x, top_boundary['sixes'], left=top_boundary['fours'], label='Sixes', color='darkred')
ax1.set_yticks(x)
ax1.set_yticklabels(top_boundary['player'])
ax1.set_xlabel('Number of Boundaries', fontsize=12)
ax1.set_title('Top Boundary Hitters - Fours vs Sixes', fontsize=14, fontweight='bold')
ax1.legend()
ax1.invert_yaxis()

# Boundary percentage distribution
ax2.hist(qualified['boundary_pct'], bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax2.axvline(qualified['boundary_pct'].median(), color='red', linestyle='--', 
           label=f"Median: {qualified['boundary_pct'].median():.1f}%")
ax2.set_xlabel('Boundary Runs % of Total', fontsize=12)
ax2.set_ylabel('Number of Players', fontsize=12)
ax2.set_title('Distribution of Boundary Percentage', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Consistency Analysis

In [ ]:
# Most consistent batsmen (high average, low dismissals per match)
qualified['dismissals_per_match'] = (qualified['dismissals'] / qualified['matches']).round(2)
qualified['runs_per_match'] = (qualified['runs'] / qualified['matches']).round(1)

print("🎯 Most Consistent Batsmen (min 200 runs, sorted by average)")
print("=" * 80)
consistent = qualified.nlargest(15, 'average')[['player', 'runs', 'matches', 'average', 'strike_rate', 'runs_per_match', 'dismissals_per_match']]
print(consistent.to_string(index=False))

In [ ]:
# Player categories based on performance
qualified['category'] = 'Average'
qualified.loc[(qualified['strike_rate'] > 140) & (qualified['average'] > 35), 'category'] = 'Elite'
qualified.loc[(qualified['strike_rate'] > 130) | (qualified['average'] > 30), 'category'] = 'Good'

print("\n📊 Player Categories:")
print(qualified['category'].value_counts())
print("\n🌟 Elite Players (SR > 140 & Avg > 35):")
print(qualified[qualified['category'] == 'Elite'][['player', 'runs', 'average', 'strike_rate']].to_string(index=False))

## 4. Key Insights & Summary

In [ ]:
print("="*80)
print("🏆 KEY BATTING INSIGHTS - T20 WORLD CUPS (2014-2024)")
print("="*80)

# Overall stats
print(f"\n📈 Overall Statistics:")
print(f"  Total runs scored: {all_deliveries['runs_total'].sum():,}")
print(f"  Total boundaries: {len(all_deliveries[all_deliveries['runs_batter'].isin([4,6])])}")
print(f"  Average score per match: {all_deliveries.groupby('match_id')['runs_total'].sum().mean():.1f}")
print(f"  Overall strike rate: {(all_deliveries['runs_batter'].sum() / len(all_deliveries) * 100):.2f}")

# Top performer
top_scorer = batting_stats.iloc[0]
print(f"\n👑 Leading Run Scorer:")
print(f"  {top_scorer['player']}: {top_scorer['runs']} runs in {top_scorer['matches']} matches")
print(f"  Average: {top_scorer['average']} | Strike Rate: {top_scorer['strike_rate']}")

# Best strike rate
best_sr = qualified.nlargest(1, 'strike_rate').iloc[0]
print(f"\n⚡ Highest Strike Rate (min 200 runs):")
print(f"  {best_sr['player']}: {best_sr['strike_rate']} SR ({best_sr['runs']} runs)")

# Most sixes
most_sixes = qualified.nlargest(1, 'sixes').iloc[0]
print(f"\n💥 Most Sixes:")
print(f"  {most_sixes['player']}: {int(most_sixes['sixes'])} sixes")

print("\n" + "="*80)

## Next Steps

- Analyze powerplay vs death overs performance
- Compare performance by tournament year
- Venue-specific batting analysis
- Team batting comparisons